In [38]:
# import pandas
import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Bar, Line

In [39]:
# import dataframes
avg_df = pd.read_pickle('files\\avg_df.pkl')
mcap_df = pd.read_pickle('files\\mcap_df.pkl')


# convert timestamp to datetime
avg_df['month'] = avg_df['timestamp'].dt.month 

In [40]:
from pyecharts import options as opts
from pyecharts.charts import Line
from pyecharts.commons.utils import JsCode

# 1) Your data
dates = [...]      # list of datetime strings or x-axis labels
values = [...]     # corresponding large numbers

# 2) JS function to abbreviate numbers
abbr_formatter = JsCode("""
function (value) {
    if (value >= 1e12) { return +(value/1e12).toFixed(1) + 'T'; }
    if (value >= 1e9)  { return +(value/1e9).toFixed(1)  + 'B'; }
    if (value >= 1e6)  { return +(value/1e6).toFixed(1)  + 'M'; }
    if (value >= 1e3)  { return +(value/1e3).toFixed(1)  + 'K'; }
    return value;
}
""")

In [41]:
# Plot average price for each symbol using pyecharts
fig = Line(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",
        height="535px",  
    )
)

# Prepare x-axis as unique date strings for plotting
x_axis = avg_df['timestamp'].sort_values().unique()
x_axis_str = [ts.strftime('%Y-%m-%d') for ts in x_axis]
fig.add_xaxis(x_axis_str)

# Add a line for each symbol, hide data labels
for symbol, group in avg_df.groupby('symbols'):
    group = group.set_index('timestamp')
    y_axis = [group['average_price'].get(ts, None) for ts in x_axis]
    fig.add_yaxis(
        symbol, y_axis,
        label_opts=opts.LabelOpts(is_show=False)
    )

# Add responsive rendering options
fig.set_global_opts(
    title_opts=opts.TitleOpts(title="Price History (Log Scale)"),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
    datazoom_opts=opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(type_="log"),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left="right"),
)

# Render for embedding with a template that includes viewport meta tag
fig.render('echarts\\embed_chart.html', template_name="simple_chart.html")
fig.render_notebook()  # For Jupyter Notebook display

C:\Users\Matheus\AppData\Local\Temp\ipykernel_32404\3071831935.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for symbol, group in avg_df.groupby('symbols'):


In [42]:
# Filter the dataframe for only BTC entries
btc_df = avg_df[avg_df['symbols'] == 'BTC'].sort_values('timestamp')

# Prepare x-axis as unique date strings for plotting
x_axis = btc_df['timestamp'].unique()
x_axis_str = [ts.strftime('%Y-%m-%d') for ts in x_axis]

# Create a new Line chart for BTC
btc_fig = Line(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",
        height="535px",
    )
)

btc_fig.add_xaxis(x_axis_str)

# Set the timestamp as the index for easier data retrieval
btc_df = btc_df.set_index('timestamp')
# Retrieve the average_price for each timestamp in x_axis
y_axis = [btc_df['average_price'].get(ts, None) for ts in x_axis]

btc_fig.add_yaxis(
    "BTC", y_axis,
    label_opts=opts.LabelOpts(is_show=False),
    areastyle_opts=opts.AreaStyleOpts(
        opacity=0.5,
        color={
            "type": "linear",
            "x": 0,
            "y": 0,
            "x2": 0,
            "y2": 1,
            "colorStops": [
                {"offset": 0, "color": "rgba(0, 0, 255, 0.5)"},
                {"offset": 1, "color": "rgba(0, 0, 0, 0)"}     
            ],
            "globalCoord": False
        }
    )
)

btc_fig.set_global_opts(
    title_opts=opts.TitleOpts(title="BTC Price History"),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
    datazoom_opts=opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
    yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(formatter=abbr_formatter)),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left="right"),
)

# Render for embedding and for notebook display
btc_fig.render('echarts\\embed_chart_btc.html', template_name="simple_chart.html")
btc_fig.render_notebook()

In [43]:
# Prepare x-axis as unique date strings for plotting from the DataFrame index
x_axis = mcap_df.index.sort_values().unique()
x_axis_str = [ts.strftime('%Y-%m-%d') for ts in x_axis]

# Retrieve y-axis data for each market cap type aligned with x_axis
total2_y = mcap_df['total2_mcap'].loc[x_axis].tolist()
btc_y = mcap_df['btc_mcap'].loc[x_axis].tolist()


# Create a new Line chart for BTC
fig = Line(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",
        height="535px",
    )
)

fig.add_xaxis(x_axis_str)

# Add the BTC Market Cap line
fig.add_yaxis(
    "BTC", btc_y,
    label_opts=opts.LabelOpts(is_show=False),
)

# Add the Total2 Market Cap line
fig.add_yaxis(
    "TOTAL2", total2_y,
    label_opts=opts.LabelOpts(is_show=False),
)

# Define the flag to switch between log and normal scale
is_log = False  # Set to False for normal (linear) scale

if is_log:
    # Log scale w/ number formatter
    yaxis_setting = opts.AxisOpts(type_="log", axislabel_opts=opts.LabelOpts(formatter=abbr_formatter))
else:
    # Linear scale w/ number formatter
    yaxis_setting = opts.AxisOpts(axislabel_opts=opts.LabelOpts(formatter=abbr_formatter))  

fig.set_global_opts(
    title_opts=opts.TitleOpts(title="BTC vs TOTAL2 Market Cap"),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
    datazoom_opts=opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
    yaxis_opts=yaxis_setting,
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
    legend_opts=opts.LegendOpts(pos_left="right", is_show=False),
)


# Render for embedding and for notebook display
fig.render('echarts\\crypto_mcap_history.html', template_name="simple_chart.html")
fig.render_notebook()

In [45]:
web3_tx = pd.read_pickle('files\\web3_tx.pkl')

# Ensure date column is in datetime, if needed
web3_tx['date'] = pd.to_datetime(web3_tx['date'])

# Get unique sorted dates as x-axis labels
dates = sorted(web3_tx['date'].unique())
x_axis = [d.strftime('%Y-%m-%d') for d in dates]

# Define the color mapping for chains
color_map = {
    'bitcoin': '#FC922F',   # Bitcoin Orange
    'ethereum': '#626AFF',  # Ethereum Blue
    'bnb_smart': '#FFCF3D',  # Binance Yellow
    'bnb_beacon': '#786834',  # Binance Yellow
    'ripple': '#DCDCDC',    # Ripple Grey
    'avalanche': '#FF3A3A',  # Avalanche Red
}

# Get unique chains and prepare data series for each
chains = web3_tx['chain'].unique()
data_series = {}

for chain in chains:
    # Filter rows for a given chain, using date as index
    df_chain = web3_tx[web3_tx['chain'] == chain].set_index('date')['tx_count']
    # Build a list of transaction counts aligned to the x_axis dates (fill missing as 0)
    series = [int(df_chain.get(d, 0)) for d in dates]
    data_series[chain] = series

# Create the stacked bar chart with pyecharts
bar = Bar(
    init_opts=opts.InitOpts(
        theme="dark",
        bg_color="rgba(0,0,0,0)",
        width="100%",
        height="535px",
    )
)

bar.add_xaxis(x_axis)

for chain, series in data_series.items():
    # Provide a custom item style if the chain exists in our color map
    itemstyle = None
    if chain in color_map:
        itemstyle = opts.ItemStyleOpts(color=color_map[chain])
    bar.add_yaxis(
        chain,
        series,
        stack="stack1",
        label_opts=opts.LabelOpts(is_show=False),
        itemstyle_opts=itemstyle,
    )

bar.set_global_opts(
    title_opts=opts.TitleOpts(title="Blockchain Monthly Transactions"),
    tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="shadow"),
    datazoom_opts=opts.DataZoomOpts(type_="slider", range_start=0, range_end=100),
    legend_opts=opts.LegendOpts(pos_left="right", is_show=False),
    yaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(formatter=abbr_formatter)),
    xaxis_opts=opts.AxisOpts(splitline_opts=opts.SplitLineOpts(is_show=False)),
)

bar.render('echarts\\blockchain_activity.html', template_name="simple_chart.html")
bar.render_notebook()  # For Jupyter Notebook display